任务 1：数据加载与概览
* 读取 CSV 文件，解析日期时间列，设为索引。
* 使用 info()、describe() 查看数据基本情况，检查缺失值（原始数据中可能有 ? 表示的缺失值，需转换为 NaN）。
* 对缺失值进行插值（interpolate）或删除。

In [ ]:
# ---------- 全局导入与设置 ----------
import io
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

DATASET_PATH = r"data/household_power_consumption.txt"
LAG_HOURS = 6
TARGET_COLUMN = "Global_active_power"

# ==================== 任务1：数据加载与概览 ====================
def task1_load_and_overview():
    raw_data = pd.read_csv(
        DATASET_PATH,
        sep=";",
        na_values=["?"],
        low_memory=False,
    )
    raw_data["Datetime"] = pd.to_datetime(
        raw_data["Date"] + " " + raw_data["Time"],
        dayfirst=True,
        errors="coerce",
    )
    raw_data = raw_data.drop(columns=["Date", "Time"])
    raw_data = raw_data.set_index("Datetime")
    raw_data = raw_data.sort_index()
    for column_name in raw_data.columns:
        raw_data[column_name] = pd.to_numeric(raw_data[column_name], errors="coerce")

    print("=" * 60)
    print("任务1：数据加载与概览")
    print("=" * 60)

    print(f"\n原始数据形状：{raw_data.shape}（时间点数 × 变量数）")

    info_buffer = io.StringIO()
    raw_data.info(buf=info_buffer)
    info_text = info_buffer.getvalue()
    print("\n>>> df.info() 输出：")
    print(info_text)

    describe_text = raw_data.describe().to_string()
    print("\n>>> df.describe() 输出：")
    print(describe_text)

    raw_missing = raw_data.isna().sum()
    print("\n>>> 原始数据缺失值统计（每列）：")
    print(raw_missing)
    print(f"总缺失值数量: {raw_missing.sum()}")

    cleaned_data = raw_data.interpolate(method="time")
    cleaned_data = cleaned_data.ffill().bfill()
    cleaned_data = cleaned_data.dropna()

    cleaned_missing = cleaned_data.isna().sum()
    print(f"\n清洗后数据形状：{cleaned_data.shape}")
    print(f"清洗后缺失值总数: {cleaned_missing.sum()}（应为0）")

    hourly_data = cleaned_data.resample("h").mean(numeric_only=True)
    hourly_data = hourly_data.interpolate(method="time").ffill().bfill()
    print(f"\n小时级数据形状：{hourly_data.shape}")

    hourly_series = hourly_data[TARGET_COLUMN].copy()
    print(f"小时级目标序列长度: {len(hourly_series)}")
    print(f"\n目标序列前5条预览：\n{hourly_series.head()}")

    return raw_data, info_text, describe_text, raw_missing, cleaned_data, hourly_data, hourly_series

(raw_data, info_text, describe_text, raw_missing,
 cleaned_data, hourly_data, hourly_series) = task1_load_and_overview()

任务 2：时序基础分析
* 绘制全局有功功率随时间变化的折线图，观察整体趋势和日周期。
* 计算日平均功率，绘制箱线图观察每日用电模式。
* 计算自相关（ACF）和偏自相关（PACF），判断序列的平稳性（可简单使用 ADF 检验）。

In [ ]:
# ==================== 任务2：时序基础分析 ====================
def task2_basic_time_series_analysis(hourly_series):
    print("=" * 60)
    print("任务2：时序基础分析")
    print("=" * 60)

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(hourly_series.index, hourly_series.values, color="tab:blue", linewidth=0.8)
    ax.set_title("全局有功功率小时序列")
    ax.set_xlabel("时间")
    ax.set_ylabel("功率（kW）")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

    daily_average = hourly_series.resample("D").mean().to_frame(name="日平均功率")
    weekday_map = {
        0: "周一", 1: "周二", 2: "周三", 3: "周四",
        4: "周五", 5: "周六", 6: "周日",
    }
    daily_average["星期几"] = daily_average.index.dayofweek.map(weekday_map)

    fig, ax = plt.subplots(figsize=(10, 6))
    daily_average.boxplot(column="日平均功率", by="星期几", ax=ax)
    ax.set_title("不同星期的日平均功率箱线图")
    ax.set_xlabel("星期")
    ax.set_ylabel("日平均功率（kW）")
    fig.suptitle("")
    fig.tight_layout()
    plt.show()

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    plot_acf(hourly_series, lags=72, ax=axes[0])
    axes[0].set_title("全局有功功率 ACF（自相关函数）")
    plot_pacf(hourly_series, lags=72, ax=axes[1], method="ywm")
    axes[1].set_title("全局有功功率 PACF（偏自相关函数）")
    fig.tight_layout()
    plt.show()

    adf_result = adfuller(hourly_series)
    adf_summary = {
        "ADF统计量": adf_result[0],
        "p值": adf_result[1],
        "滞后阶数": adf_result[2],
        "样本量": adf_result[3],
    }
    print("\n>>> ADF平稳性检验结果：")
    for key, value in adf_summary.items():
        print(f"  {key}: {value}")
    if adf_result[1] < 0.05:
        print("  结论：p值 < 0.05，序列是平稳的（拒绝存在单位根的零假设）")
    else:
        print("  结论：p值 >= 0.05，序列是非平稳的（不能拒绝存在单位根的零假设）")

    return daily_average, adf_summary

daily_average, adf_summary = task2_basic_time_series_analysis(hourly_series)

任务 3：时序分解
* 使用 statsmodels.tsa.seasonal.seasonal_decompose 将时间序列分解为趋势、季节性和残差三部分。
* 绘制趋势、季节性和残差的分解图，分析用电量的日/周周期性（例如工作日与周末差异）。

In [ ]:
# ==================== 任务3：时序分解 ====================
def task3_time_series_decomposition(hourly_series):
    print("=" * 60)
    print("任务3：时序分解")
    print("=" * 60)

    decomposition = seasonal_decompose(hourly_series, model="additive", period=24)
    fig = decomposition.plot()
    fig.set_size_inches(14, 10)
    fig.tight_layout()
    plt.show()

    profile_data = hourly_series.to_frame(name="功率")
    profile_data["小时"] = profile_data.index.hour
    profile_data["是否周末"] = np.where(profile_data.index.dayofweek >= 5, "周末", "工作日")
    profile_table = profile_data.groupby(["是否周末", "小时"])["功率"].mean().unstack(0)

    fig, ax = plt.subplots(figsize=(12, 5))
    for column_name in profile_table.columns:
        ax.plot(profile_table.index, profile_table[column_name],
                marker="o", label=column_name)
    ax.set_title("工作日与周末的小时平均功率对比")
    ax.set_xlabel("小时")
    ax.set_ylabel("平均功率（kW）")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    workday_avg = float(profile_data[profile_data["是否周末"] == "工作日"]["功率"].mean())
    weekend_avg = float(profile_data[profile_data["是否周末"] == "周末"]["功率"].mean())
    print(f"\n用电量日/周周期分析：")
    print(f"  工作日平均功率: {workday_avg:.4f} kW")
    print(f"  周末平均功率: {weekend_avg:.4f} kW")

    return decomposition, profile_table

decomposition, profile_table = task3_time_series_decomposition(hourly_series)

任务 4：离群点检测——基于统计方法
* 使用 Z-score 方法检测全局有功功率的离群点（设定阈值为 3 倍标准差）。
* 标记离群点并绘制在原始序列上，观察异常发生的时间段。

In [ ]:
# ==================== 任务4：离群点检测——基于统计方法 ====================
def task4_outlier_zscore(hourly_series, threshold=3.0):
    print("=" * 60)
    print("任务4：离群点检测——基于统计方法（Z-score）")
    print("=" * 60)

    z_score = (hourly_series - hourly_series.mean()) / hourly_series.std(ddof=0)
    outlier_flag = (np.abs(z_score) > threshold).astype(int)
    outlier_points = hourly_series[outlier_flag == 1]

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(hourly_series.index, hourly_series.values,
            color="tab:blue", linewidth=0.8, label="原始序列")
    ax.scatter(
        outlier_points.index,
        outlier_points.values,
        color="red",
        s=20,
        label=f"Z-score离群点 (|Z|>{threshold})",
    )
    ax.set_title(f"基于 Z-score 的离群点检测（阈值={threshold}）")
    ax.set_xlabel("时间")
    ax.set_ylabel("功率（kW）")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    outlier_count = outlier_flag.sum()
    total_count = len(hourly_series)
    print(f"\nZ-score离群点检测结果（阈值={threshold}）：")
    print(f"  离群点数量: {outlier_count}")
    print(f"  总样本数: {total_count}")
    print(f"  离群点占比: {outlier_count / total_count * 100:.2f}%")
    print(f"\n  离群值最大的5个时间点：")
    top_outliers = np.abs(z_score).sort_values(ascending=False).head(5)
    for time_point in top_outliers.index:
        print(f"    {time_point}: 功率={hourly_series[time_point]:.4f}, Z-score={z_score[time_point]:.4f}")

    return z_score, outlier_flag

z_score, zscore_flag = task4_outlier_zscore(hourly_series, threshold=3.0)

任务 5：离群点检测——基于模型方法
* 使用 孤立森林（IsolationForest）对时序特征（如最近 6 小时的功率均值、方差）进行离群点检测。
* 对比 Z-score 与孤立森林的检测结果，讨论各自的适用场景。

In [ ]:
# ==================== 任务5：离群点检测——基于模型方法 ====================
def build_iforest_features(hourly_series, window_size):
    feature_data = pd.DataFrame(index=hourly_series.index)
    feature_data["current_power"] = hourly_series
    feature_data["mean_6h"] = hourly_series.rolling(window=window_size).mean()
    feature_data["std_6h"] = hourly_series.rolling(window=window_size).std()
    feature_data["min_6h"] = hourly_series.rolling(window=window_size).min()
    feature_data["max_6h"] = hourly_series.rolling(window=window_size).max()
    feature_data["diff_1h"] = hourly_series.diff(1)
    feature_data = feature_data.dropna()
    return feature_data

def task5_outlier_iforest(hourly_series, zscore_flag, window_size=6):
    print("=" * 60)
    print("任务5：离群点检测——基于模型方法（孤立森林）")
    print("=" * 60)

    feature_data = build_iforest_features(hourly_series, window_size)
    model = IsolationForest(
        n_estimators=200,
        contamination=0.01,
        random_state=42,
    )
    prediction = model.fit_predict(feature_data)
    iforest_flag_compact = pd.Series(
        np.where(prediction == -1, 1, 0),
        index=feature_data.index,
        name="iforest_outlier",
    )
    aligned_iforest_flag = pd.Series(0, index=hourly_series.index, name="iforest_outlier")
    aligned_iforest_flag.loc[iforest_flag_compact.index] = iforest_flag_compact

    outlier_points = hourly_series[aligned_iforest_flag == 1]
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(hourly_series.index, hourly_series.values,
            color="tab:blue", linewidth=0.8, label="原始序列")
    ax.scatter(
        outlier_points.index,
        outlier_points.values,
        color="darkorange",
        s=20,
        label="孤立森林离群点",
    )
    ax.set_title("基于孤立森林的离群点检测")
    ax.set_xlabel("时间")
    ax.set_ylabel("功率（kW）")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    comparison_data = pd.DataFrame(index=zscore_flag.index)
    comparison_data["zscore"] = zscore_flag.astype(int)
    comparison_data["iforest"] = aligned_iforest_flag.astype(int)
    overlap_count = int(((comparison_data["zscore"] == 1) & (comparison_data["iforest"] == 1)).sum())
    comparison_summary = {
        "Z-score离群点数量": int(comparison_data["zscore"].sum()),
        "孤立森林离群点数量": int(comparison_data["iforest"].sum()),
        "共同识别数量": overlap_count,
        "仅Z-score识别": int(comparison_data["zscore"].sum()) - overlap_count,
        "仅孤立森林识别": int(comparison_data["iforest"].sum()) - overlap_count,
    }

    print("\n>>> 离群点检测结果对比：")
    for key, value in comparison_summary.items():
        print(f"  {key}: {value}")
    print("\n方法对比分析：")
    print("  Z-score：基于正态分布假设，适合数据近似正态分布的场景，")
    print("           计算简单、可解释性强，但对非对称分布或长尾数据效果差。")
    print("  孤立森林：不依赖分布假设，能捕获多维特征中的复杂异常模式，")
    print("           适合非线性和多特征场景，但需要合理的参数调优。")

    return aligned_iforest_flag, feature_data, comparison_summary

iforest_flag, iforest_feature_data, comparison_summary = task5_outlier_iforest(
    hourly_series, zscore_flag, window_size=LAG_HOURS
)

combined_anomaly_flag = ((zscore_flag == 1) | (iforest_flag == 1)).astype(int)

任务 6：构造监督学习数据集
* 将时序预测问题转化为监督学习：利用过去 n 个时间步（例如 lag=6 小时）的全局有功功率预测下一时刻的功率。
* 添加时间特征：小时、星期几、是否为周末等作为额外输入。
* 划分训练集（前 80%）和测试集（后 20%），注意保持时序顺序，不能随机打乱。

In [ ]:
# ==================== 任务6：构造监督学习数据集 ====================
def task6_build_supervised_dataset(hourly_series, lag_hours, anomaly_flag):
    print("=" * 60)
    print("任务6：构造监督学习数据集")
    print("=" * 60)

    supervised_data = pd.DataFrame(index=hourly_series.index)
    supervised_data["target"] = hourly_series
    for lag_step in range(1, lag_hours + 1):
        supervised_data[f"lag_{lag_step}"] = hourly_series.shift(lag_step)
    supervised_data["hour"] = supervised_data.index.hour
    supervised_data["day_of_week"] = supervised_data.index.dayofweek
    supervised_data["is_weekend"] = (supervised_data.index.dayofweek >= 5).astype(int)
    supervised_data["month"] = supervised_data.index.month
    supervised_data["anomaly_flag"] = anomaly_flag.reindex(supervised_data.index).fillna(0).astype(int)
    supervised_data = supervised_data.dropna()
    print(f"监督学习数据集形状: {supervised_data.shape}（样本数 × 特征数+目标）")

    feature_columns = [col for col in supervised_data.columns if col != "target"]
    features = supervised_data[feature_columns].copy()
    target = supervised_data["target"].copy()
    print(f"特征矩阵形状: {features.shape}，包含 {len(feature_columns)} 个特征")
    print(f"目标向量形状: {target.shape}")

    train_ratio = 0.8
    split_index = int(len(features) * train_ratio)
    x_train = features.iloc[:split_index].copy()
    y_train = target.iloc[:split_index].copy()
    x_test = features.iloc[split_index:].copy()
    y_test = target.iloc[split_index:].copy()

    print(f"\n训练集样本数: {len(x_train)}（{len(x_train)/len(features)*100:.1f}%）")
    print(f"测试集样本数: {len(x_test)}（{len(x_test)/len(features)*100:.1f}%）")
    print(f"训练集时间范围: {x_train.index[0]} ~ {x_train.index[-1]}")
    print(f"测试集时间范围: {x_test.index[0]} ~ {x_test.index[-1]}")

    return supervised_data, features, target, x_train, x_test, y_train, y_test

(supervised_data, features, target,
 x_train, x_test, y_train, y_test) = task6_build_supervised_dataset(
    hourly_series, LAG_HOURS, combined_anomaly_flag
)

任务 7：数据标准化与离群点处理
* 对特征进行 Z-Score 标准化（StandardScaler）。
* 可尝试剔除训练集中的离群点（使用上一阶段检测出的异常标签），观察对模型预测的影响。

In [ ]:
# ==================== 任务7：数据标准化与离群点处理 ====================
def task7_standardize_and_clean(x_train, x_test, y_train):
    print("=" * 60)
    print("任务7：数据标准化与离群点处理")
    print("=" * 60)

    scaler_all = StandardScaler()
    x_train_scaled_all = scaler_all.fit_transform(x_train)
    x_test_scaled_all = scaler_all.transform(x_test)

    print("\n>>> 全量数据标准化结果：")
    print(f"  训练集各特征均值范围: [{x_train_scaled_all.mean(axis=0).min():.6f}, {x_train_scaled_all.mean(axis=0).max():.6f}]")
    print(f"  训练集各特征标准差范围: [{x_train_scaled_all.std(axis=0).min():.6f}, {x_train_scaled_all.std(axis=0).max():.6f}]")

    filtered_mask = x_train["anomaly_flag"] == 0
    filtered_x_train = x_train.loc[filtered_mask].copy()
    filtered_y_train = y_train.loc[filtered_mask].copy()
    print(f"\n>>> 剔除离群点：训练集从 {len(x_train)} 缩减至 {len(filtered_x_train)} 样本")
    print(f"  移除样本数: {len(x_train) - len(filtered_x_train)}")

    scaler_filtered = StandardScaler()
    filtered_x_train_scaled = scaler_filtered.fit_transform(filtered_x_train)
    x_test_scaled_filtered = scaler_filtered.transform(x_test)

    print(f"\n>>> 过滤后数据标准化结果：")
    print(f"  过滤后训练集各特征均值范围: [{filtered_x_train_scaled.mean(axis=0).min():.6f}, {filtered_x_train_scaled.mean(axis=0).max():.6f}]")
    print(f"  过滤后训练集各特征标准差范围: [{filtered_x_train_scaled.std(axis=0).min():.6f}, {filtered_x_train_scaled.std(axis=0).max():.6f}]")

    return (scaler_all, x_train_scaled_all, x_test_scaled_all,
            scaler_filtered, filtered_x_train_scaled, x_test_scaled_filtered,
            filtered_y_train)

(scaler_all, x_train_scaled_all, x_test_scaled_all,
 scaler_filtered, filtered_x_train_scaled, x_test_scaled_filtered,
 filtered_y_train) = task7_standardize_and_clean(x_train, x_test, y_train)

任务 8：构建 BP 神经网络
* 使用 sklearn.neural_network.MLPRegressor 构建多层感知机（隐藏层 2~3 层，神经元数 50~100，激活函数 relu）。
* 设置合适的超参数（max_iter、alpha 正则化项、学习率 learning_rate_init）。
* 在训练集上拟合模型，记录损失下降曲线。

In [ ]:
# ==================== 任务8：构建BP神经网络 ====================
def task8_build_mlp(x_train_scaled_all, y_train, filtered_x_train_scaled, filtered_y_train):
    print("=" * 60)
    print("任务8：构建BP神经网络（MLP）")
    print("=" * 60)

    print("\n>>> 训练基准模型（含离群点）...")
    baseline_model = MLPRegressor(
        hidden_layer_sizes=(100, 64, 32),
        activation="relu",
        solver="adam",
        alpha=0.0005,
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
    )
    baseline_model.fit(x_train_scaled_all, y_train)
    print(f"  基准模型迭代次数: {baseline_model.n_iter_}")
    print(f"  基准模型最终损失: {baseline_model.loss_:.6f}")

    print("\n>>> 训练过滤模型（剔除离群点）...")
    filtered_model = MLPRegressor(
        hidden_layer_sizes=(100, 64, 32),
        activation="relu",
        solver="adam",
        alpha=0.0005,
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
    )
    filtered_model.fit(filtered_x_train_scaled, filtered_y_train)
    print(f"  过滤模型迭代次数: {filtered_model.n_iter_}")
    print(f"  过滤模型最终损失: {filtered_model.loss_:.6f}")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(baseline_model.loss_curve_, label="基准模型（含离群点）", linewidth=1.5)
    ax.plot(filtered_model.loss_curve_, label="过滤模型（剔除离群点）", linewidth=1.5)
    ax.set_title("MLP 训练损失曲线")
    ax.set_xlabel("迭代次数")
    ax.set_ylabel("Loss（损失值）")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    return baseline_model, filtered_model

baseline_model, filtered_model = task8_build_mlp(
    x_train_scaled_all, y_train, filtered_x_train_scaled, filtered_y_train
)

任务 9：模型评估与对比
* 在测试集上预测，计算均方根误差（RMSE）、平均绝对误差（MAE）、决定系数（R²）。
* 绘制真实值与预测值的对比折线图，直观展示预测效果。

In [ ]:
# ==================== 任务9：模型评估与对比 ====================
def task9_evaluate_and_compare(
    baseline_model, x_test_scaled_all, y_test,
    filtered_model, x_test_scaled_filtered
):
    print("=" * 60)
    print("任务9：模型评估与对比")
    print("=" * 60)

    baseline_prediction = baseline_model.predict(x_test_scaled_all)
    baseline_metrics = {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, baseline_prediction))),
        "MAE": float(mean_absolute_error(y_test, baseline_prediction)),
        "R2": float(r2_score(y_test, baseline_prediction)),
    }

    filtered_prediction = filtered_model.predict(x_test_scaled_filtered)
    filtered_metrics = {
        "RMSE": float(np.sqrt(mean_squared_error(y_test, filtered_prediction))),
        "MAE": float(mean_absolute_error(y_test, filtered_prediction)),
        "R2": float(r2_score(y_test, filtered_prediction)),
    }

    print("\n>>> 模型评估指标对比：")
    print("--" * 30)
    print("基准模型（含离群点训练）：")
    print(f"  RMSE: {baseline_metrics['RMSE']:.4f} kW")
    print(f"  MAE : {baseline_metrics['MAE']:.4f} kW")
    print(f"  R²  : {baseline_metrics['R2']:.4f}")
    print("--" * 30)
    print("过滤模型（剔除离群点训练）：")
    print(f"  RMSE: {filtered_metrics['RMSE']:.4f} kW")
    print(f"  MAE : {filtered_metrics['MAE']:.4f} kW")
    print(f"  R²  : {filtered_metrics['R2']:.4f}")
    print("--" * 30)

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(y_test.index, y_test.values, color="black", linewidth=1.0, label="真实值")
    ax.plot(y_test.index, baseline_prediction, linewidth=0.9,
            label=f"基准模型预测 (R²={baseline_metrics['R2']:.4f})")
    ax.plot(y_test.index, filtered_prediction, linewidth=0.9,
            label=f"过滤模型预测 (R²={filtered_metrics['R2']:.4f})")
    ax.set_title("测试集真实值与预测值对比")
    ax.set_xlabel("时间")
    ax.set_ylabel("功率（kW）")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    metrics_summary = {
        "基准模型": baseline_metrics,
        "剔除离群点模型": filtered_metrics,
    }
    return baseline_prediction, baseline_metrics, filtered_prediction, filtered_metrics, metrics_summary

(baseline_prediction, baseline_metrics,
 filtered_prediction, filtered_metrics, metrics_summary) = task9_evaluate_and_compare(
    baseline_model, x_test_scaled_all, y_test,
    filtered_model, x_test_scaled_filtered
)

任务 10：扩展任务（可选，选做且正确加10分）
* 尝试使用更复杂的神经网络（如使用 keras 构建 LSTM）进行预测，并与 MLP 对比。
* 应用其他离群点检测算法（如 DBSCAN 或基于聚类的异常检测）并比较效果。

In [ ]:
# ==================== 任务10：扩展任务（LSTM+DBSCAN） ====================
def task10_extensions(hourly_series, feature_data_from_iforest,
                      x_train_scaled_all, x_test_scaled_all, y_train, y_test,
                      x_train, x_test):
    print("=" * 60)
    print("任务10：扩展任务（LSTM + DBSCAN）")
    print("=" * 60)

    # ----- LSTM部分 -----
    print("\n>>> 10.1 LSTM模型预测")
    try:
        from keras.models import Sequential
        from keras.layers import LSTM, Dense, Dropout
        from keras.optimizers import Adam
        from keras.callbacks import EarlyStopping

        n_train = len(x_train_scaled_all)
        n_test = len(x_test_scaled_all)
        lstm_lag_features_train = x_train_scaled_all[:, :LAG_HOURS].reshape(n_train, LAG_HOURS, 1)
        lstm_lag_features_test = x_test_scaled_all[:, :LAG_HOURS].reshape(n_test, LAG_HOURS, 1)

        lstm_model = Sequential([
            LSTM(50, return_sequences=True, input_shape=(LAG_HOURS, 1)),
            Dropout(0.2),
            LSTM(30, return_sequences=False),
            Dropout(0.2),
            Dense(1, activation="linear")
        ])
        lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
        early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=0)

        print("LSTM模型结构：")
        lstm_model.summary()

        lstm_history = lstm_model.fit(
            lstm_lag_features_train, y_train.values,
            epochs=50, batch_size=32,
            validation_split=0.1,
            callbacks=[early_stop],
            verbose=0
        )

        lstm_prediction = lstm_model.predict(lstm_lag_features_test, verbose=0).flatten()
        lstm_rmse = float(np.sqrt(mean_squared_error(y_test, lstm_prediction)))
        lstm_mae = float(mean_absolute_error(y_test, lstm_prediction))
        lstm_r2 = float(r2_score(y_test, lstm_prediction))

        print(f"\nLSTM模型评估结果：")
        print(f"  RMSE: {lstm_rmse:.4f} kW")
        print(f"  MAE : {lstm_mae:.4f} kW")
        print(f"  R²  : {lstm_r2:.4f}")

        fig, axes = plt.subplots(2, 1, figsize=(15, 10))
        axes[0].plot(y_test.index, y_test.values, color="black", linewidth=1.0, label="真实值")
        axes[0].plot(y_test.index, baseline_prediction, linewidth=0.9,
                     label=f"MLP预测 (R²={baseline_metrics['R2']:.4f})")
        axes[0].plot(y_test.index, lstm_prediction, linewidth=0.9,
                     label=f"LSTM预测 (R²={lstm_r2:.4f})")
        axes[0].set_title("MLP vs LSTM 预测对比")
        axes[0].set_ylabel("功率（kW）")
        axes[0].grid(alpha=0.3)
        axes[0].legend()

        axes[1].plot(lstm_history.history["loss"], label="训练损失", linewidth=1.5)
        axes[1].plot(lstm_history.history["val_loss"], label="验证损失", linewidth=1.5)
        axes[1].set_title("LSTM 训练/验证损失曲线")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss（MSE）")
        axes[1].grid(alpha=0.3)
        axes[1].legend()
        fig.tight_layout()
        plt.show()

        print(f"\n>>> MLP vs LSTM 对比：")
        print(f"  模型        RMSE(kW)    MAE(kW)     R²")
        print(f"  MLP基准     {baseline_metrics['RMSE']:.4f}      {baseline_metrics['MAE']:.4f}      {baseline_metrics['R2']:.4f}")
        print(f"  LSTM        {lstm_rmse:.4f}      {lstm_mae:.4f}      {lstm_r2:.4f}")
        lstm_available = True
    except ImportError:
        print("\n注意：Keras/TensorFlow未安装，跳过LSTM部分。")
        print("可使用 pip install tensorflow 安装后再运行。")
        lstm_available = False
        lstm_rmse, lstm_mae, lstm_r2 = None, None, None

    # ----- DBSCAN部分 -----
    print("\n" + "=" * 30)
    print(">>> 10.2 DBSCAN离群点检测")
    from sklearn.cluster import DBSCAN
    from sklearn.preprocessing import StandardScaler as SS2

    dbscan_scaler = SS2()
    iforest_features_scaled = dbscan_scaler.fit_transform(feature_data_from_iforest)

    dbscan = DBSCAN(eps=0.5, min_samples=10)
    dbscan_labels = dbscan.fit_predict(iforest_features_scaled)

    dbscan_flag = pd.Series(0, index=hourly_series.index, name="dbscan_outlier")
    dbscan_noise = pd.Series(
        np.where(dbscan_labels == -1, 1, 0),
        index=feature_data_from_iforest.index
    )
    dbscan_flag.loc[dbscan_noise.index] = dbscan_noise

    outlier_points_dbscan = hourly_series[dbscan_flag == 1]
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(hourly_series.index, hourly_series.values,
            color="tab:blue", linewidth=0.8, label="原始序列")
    ax.scatter(
        outlier_points_dbscan.index,
        outlier_points_dbscan.values,
        color="purple",
        s=20,
        label="DBSCAN离群点",
    )
    ax.set_title("基于 DBSCAN 的离群点检测")
    ax.set_xlabel("时间")
    ax.set_ylabel("功率（kW）")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    n_noise = int((dbscan_labels == -1).sum())
    print(f"\nDBSCAN离群点检测结果：")
    print(f"  发现的簇数量: {n_clusters}")
    print(f"  噪声点（离群点）数量: {n_noise}")
    print(f"  噪声点占比: {n_noise / len(dbscan_labels) * 100:.2f}%")

    print(f"\n>>> 三种离群点检测方法对比：")
    print(f"  Z-score离群点数量:     {int(zscore_flag.sum())}")
    print(f"  孤立森林离群点数量:     {int(iforest_flag.sum())}")
    print(f"  DBSCAN离群点数量:       {int(dbscan_flag.sum())}")
    print(f"\n方法适用场景总结：")
    print(f"  Z-score  ：适合近似正态分布、单变量检测，计算快、可解释性强")
    print(f"  孤立森林：适合多维特征、不依赖分布假设，能捕获复杂异常模式")
    print(f"  DBSCAN  ：基于密度聚类，能发现任意形状的簇，适合空间分布异常检测")

    return {
        "lstm_available": lstm_available,
        "lstm_rmse": lstm_rmse, "lstm_mae": lstm_mae, "lstm_r2": lstm_r2,
        "dbscan_n_clusters": n_clusters, "dbscan_n_noise": n_noise,
        "dbscan_flag": dbscan_flag,
    }

ext_results = task10_extensions(
    hourly_series, iforest_feature_data,
    x_train_scaled_all, x_test_scaled_all, y_train, y_test,
    x_train, x_test
)

# ============================================================
# 所有任务执行完毕
# ============================================================
print("\n" + "=" * 60)
print("实验4：所有任务执行完毕！")
print("=" * 60)
print(f"最终模型性能总结：")
print(f"  基准MLP模型: RMSE={baseline_metrics['RMSE']:.4f}, MAE={baseline_metrics['MAE']:.4f}, R²={baseline_metrics['R2']:.4f}")
print(f"  过滤MLP模型: RMSE={filtered_metrics['RMSE']:.4f}, MAE={filtered_metrics['MAE']:.4f}, R²={filtered_metrics['R2']:.4f}")
if ext_results.get("lstm_available"):
    print(f"  LSTM模型   : RMSE={ext_results['lstm_rmse']:.4f}, MAE={ext_results['lstm_mae']:.4f}, R²={ext_results['lstm_r2']:.4f}")